
# Setup Script voor RvW-tool

#### Ontwikkeld door: Wietse Wierks (HDSR), Rob Tijsen (AGV) & Rafi Senden (AGV)  
In opdracht van: Deltaprogramma Centraal Holland binnen het Maatregelenpakket Wateroverlast. 

**DISCLAIMER:**  
Dit is een werkbestand en nog in ontwikkeling. Signaleer je fouten of onduidelijkheden, neem dan contact op via: wietse.wierks@hdsr.nl  

---

## Introductie

Deze tool is ontwikkeld binnen het Deltaprogramma Centraal Holland voor het maatregelenpakket *Wateroverlast – Traject Ruimte voor Water*.  

De tool bestaat uit twee scripts:
- Voorbewerking van data die gebruikt wordt voor de tool (dit script);
- De daadwerkelijke tool waarin de RvW wordt berekend in m³ en m².

De invoer en  werking van de functies wordt toegelicht via comments en markdown cellen. In de comments in het script zelf wordt een toelichting gegeven over de werking van de functie. In de markdown cellen wordt uitgelegd per stap waarom deze functie wordt uitgevoerd. 

## Workflow

Het script is opgedeeld in **3 delen**:

---

### Deel 1 – Bepalen van rekengebied
1.1 Inladen van data  
1.2 Opzetten van een RvW-dataframe  
1.3 Selecteren van poldergebieden  

### Deel 2 – Koppelen watervlakken en peilen aan rekengebied
2.1 Koppelen van zomer- en winterpeilen aan watervlakken binnen de polder  
2.2 Rasteriseren van watervlakken met corresponderend peil  

### Deel 3 – Voorbewerking AHN rekengebied
3.1 Combineren van AHN-grondfilter met AHN-data uit de Lizard-catalogus  
3.2 Combineren van resulterende AHN met peilraster    

# Deel 1 - Bepalen van rekengebied

### 1.1 Inladen data


**Doel**  
Definiëren van de invoerdata voor het bepalen van het rekengebied en de verdere analyse.

**Datasets**
- **afvoer_geb**: afvoergebieden (peilgebieden) als shapefile  
- **pg_polygon**: polygonen van peilgebieden met oppervlakte-informatie  
- **bem_pg_polygon**: polygonen van de bemalen peilvakken gekoppeld aan afvoergebieden
- **water_vlakken**: watervlakken met geometrie van watergangen  
- **dem_lizard**: AHN raster via Lizard (dichtgesmeerd)  
- **dem_basisdata**: AHN basisdata (representatief voor oever)  

**Output**
- Paden naar datasets die worden gebruikt in verdere stappen  


In [1]:
import os
import arcpy
import pandas as pd
import numpy as np
import re
import gc
from arcpy import env
from arcpy.sa import Raster, SetNull, Int, IsNull
from arcpy.sa import *
from arcpy.sa import ExtractByMask, FocalStatistics, Int


basis_pad = r"C:/Users/Senden02/OneDrive - Waternet Amsterdam/Documenten/ArcGIS/Projects/RvW-DPCH/rvw_tool"
os.chdir(basis_pad)

arcpy.env.workspace = basis_pad
arcpy.env.overwriteOutput = True

In [2]:
afvoer_geb       = r"C:\Users\Senden02\OneDrive - Waternet Amsterdam\Documenten\ArcGIS\Projects\RvW-DPCH\rvw_tool\02_data_raw\Peilgebieden\Centraal Holland\afvoergebieden_compleet_correct_dp_hout.shp"
pg_polygon       = r"C:\Users\Senden02\OneDrive - Waternet Amsterdam\Documenten\ArcGIS\Projects\RvW-DPCH\rvw_tool\03_data_intermediate\peilgebieden_compleet_opper_correct_dp_hout.shp"
bem_pg_polygon   = r"C:\Users\Senden02\OneDrive - Waternet Amsterdam\Documenten\ArcGIS\Projects\RvW-DPCH\rvw_tool\03_data_intermediate\bem_pg_selectie.shp"
water_vlakken    = r"C:\Users\Senden02\OneDrive - Waternet Amsterdam\Documenten\ArcGIS\Projects\RvW-DPCH\rvw_tool\01_src\03_data_processing\ahn_merger\ahn_merger.gdb\watervlakken_agv"
dem_lizard       = r"C:\Users\Senden02\OneDrive - Waternet Amsterdam\Documenten\ArcGIS\Projects\RvW-DPCH\rvw_tool\02_data_raw\AHN\AHN_Lizard\ahn4_dpch_int_tiff\ahn4_dpch_int.tif"
dem_basisdata    = r"D:\01_data\01_AHN\AHN_basisdata_waterschappen\AGV_Data\AHN4_DTM_AGV\ahn4_agv_x10"

### 1.2 Opzetten RvW-dataframe

**Doel**  
Opzetten van een dataframe waarin afvoergebieden en bemalen peilgebieden worden gecombineerd.

**Werking**
- Inlezen van afvoergebieden  
- Inlezen van bemalen peilgebieden  
- Groeperen van peilgebieden per polder  
- Samenvoegen tot één dataframe  

**Output**
- Dataframe met per polder:
  - kenmerken afvoergebied  
  - gekoppelde peilgebieden  





### 1.2 Achtergrondfunctie: opzetten RvW-dataframe

Deze functie combineert verschillende GIS-datasets tot één analyseklaar dataframe.
De koppeling gebeurt via <b>WS_ID</b> en peilgebieden worden geaggregeerd per polder.

In [3]:
def build_rvw_dataframe(
    afvoer_fc,
    bemalen_pg_fc,
    pg_polygon,
    *,
    afvoer_fields,
    afvoer_rename_map,
    pg_fields,
    join_field="WS_ID",
    pg_group_field="WS_ID",
    pg_list_field="CODE",
    pg_list_name="Bemalen PG Code",
    output_columns=None
):
    """
    Bouwt een overzichtsdataframe waarin afvoergebieden worden verrijkt met 
    lijsten van bemalen peilgebieden en alle peilgebieden binnen dezelfde polder.
    
    Parameters
    ----------
    afvoer_fc : str
        Pad naar feature class met afvoer/poldergebieden
    pg_polygon : str
        Pad naar feature class met alle peilgebiedpolygonen    
    bemalen_pg_fc : str
        Pad naar feature class met bemalen peilgebieden gekoppeld aan afvoergebieden
    afvoer_fields : list[str]
        Velden die uit afvoer_fc gelezen worden
    afvoer_rename_map : dict
        Mapping van inputveld -> outputveldnaam
    pg_fields : list[str]
        Velden die uit bemalen_pg_fc gelezen worden
    join_field : str
        Veld waarop de merge plaatsvindt
    pg_group_field : str
        Veld waarop gegroepeerd wordt (meestal gelijk aan join_field)
    pg_list_field : str
        Veld dat als lijst wordt geaggregeerd
    pg_list_name : str
        Naam van de gegenereerde lijstkolom
    output_columns : list[str], optional
        Volgorde en selectie van outputkolommen

    Returns
    -------
    pandas.DataFrame
    DataFrame met afvoergebieden, een lijst van bemalen peilgebieden en een 
    lijst van alle peilgebieden binnen de polder.
    """

    # --- Afvoergebieden ---
    afvoer_data = []
    with arcpy.da.SearchCursor(afvoer_fc, afvoer_fields) as cursor:
        for row in cursor:
            afvoer_data.append(row)

    df_afvoer = pd.DataFrame(afvoer_data, columns=afvoer_fields)
    df_afvoer = df_afvoer.rename(columns=afvoer_rename_map)

    # --- Bemalen peilgebieden ---
    pg_data = []
    with arcpy.da.SearchCursor(bemalen_pg_fc, pg_fields) as cursor:
        for row in cursor:
            pg_data.append(row)

    df_pg = pd.DataFrame(pg_data, columns=pg_fields)

    df_pg_codes = (
        df_pg
        .groupby(pg_group_field)[pg_list_field]
        .apply(list)
        .reset_index(name=pg_list_name)
    )
    
    # --- Peilgebieden polygonen ---
    pg_poly_data = []
    with arcpy.da.SearchCursor(pg_polygon, pg_fields) as cursor:
        for row in cursor:
            pg_poly_data.append(row)

    df_pg_poly = pd.DataFrame(pg_poly_data, columns=pg_fields)

    df_pg_poly_codes = (
        df_pg_poly
        .groupby(pg_group_field)["CODE"]
        .apply(list)
        .reset_index(name="Peilgebieden in polder")
    )

    # --- Merge ---
    df_out = df_afvoer.merge(
        df_pg_codes,
        on=join_field,
        how="left"
    )
        
    df_out = df_out.merge(
        df_pg_poly_codes,
        on=join_field,
        how="left"
    )

    # --- Kolomselectie ---
    if output_columns:
        df_out = df_out[output_columns]
        
    df_out  

    return df_out


### 1.2 Invoer: configuratie RvW-dataframe

Hier wordt bepaald welke kolommen worden gebruikt en hoe deze worden benoemd.


In [4]:
# Kolommen uit afvoergebieden shapefile die in dataframe verwerkt moeten worden
afvoer_fields = ["Naam_1", "Waterschap", "WS_ID", "oppervlakt", "SHAPE"]

# Geef aan welke kolommen hernoemt moeten worden
afvoer_rename_map = {
    "Naam_1": "Polder"
}

# Kolommen uit bemalen peilgebieden shapefile
pg_fields = [
    "CODE", "Peil_wi", "Peil_zo",
    "Type", "Waterschap", "Naam_1", "WS_ID"]

# Roep dataframe functie op
df_rvw = build_rvw_dataframe(
    afvoer_fc=afvoer_geb,
    bemalen_pg_fc=bem_pg_polygon,
    pg_polygon=pg_polygon,
    afvoer_fields=afvoer_fields,
    afvoer_rename_map=afvoer_rename_map,
    pg_fields=pg_fields,
    join_field="WS_ID",
    pg_group_field="WS_ID",
    pg_list_field="CODE",
    pg_list_name="Bemalen PG Code",
    output_columns=["WS_ID", "Polder", "Waterschap", 
        "Bemalen PG Code", "Peilgebieden in polder", "oppervlakt", "SHAPE"]
)

df_rvw

,WS_ID,Polder,Waterschap,Bemalen PG Code,Peilgebieden in polder,oppervlakt,SHAPE
0,AGV - 393,Polder Holland en Sticht west,AGV,[39.2-4],"[39.2-2, 39.2-3, 39.2-4, 39.2-1, 39.2-5, 39.2-6]",2.248936e+06,"(128405.66255119733, 469886.6168413003)"
1,AGV - 395,Westerpark,AGV,NaN,[3693-2],6.325240e+04,"(120339.08548812404, 488865.4387489665)"
2,AGV - 396,Gansenhoef west,AGV,NaN,[65-2],3.644121e+05,"(130166.59784773768, 463321.3893446155)"
3,AGV - 397,Breukelen boezempeil,AGV,NaN,[6580-01],1.738017e+05,"(128471.82329957125, 464685.9540106039)"
4,AGV - 402,De Lange Bretten,AGV,NaN,"[3695-3, 3695-2, 3695-1, 3695-4]",2.424393e+06,"(114872.00875706122, 488955.2822263788)"
...,...,...,...,...,...,...,...
511,Rijnland - 401,Het Amsterdamse Bos,Rijnland,[PBS_GH-190.00],"[PBS_GH-190.01.1, PBS_GH-190.00, PBS_GH-190.02...",7.263430e+06,"(117132.20695113626, 480852.600821202)"
512,Rijnland - 408,Polder Marendijk,Rijnland,NaN,[PBS_RL-011],0.000000e+00,"(93523.97765777455, 464802.773163694)"
513,Rijnland - 107,"Oostvliet-, Hof- en Spekpolder",Rijnland,[PBS_WW-03A],"[PBS_WW-03B, PBS_WW-03C, PBS_WW-03A]",2.465940e+06,"(92844.78663406642, 460564.18128035014)"
514,Rijnland - 231,Ridderveld en de Bijlen,Rijnland,[PBS_OR-4.15.1.1],"[PBS_OR-4.15.1.2, PBS_OR-4.15.1.1, PBS_OR-4.15...",4.024068e+06,"(105892.47600625294, 461889.5084060144)"


### 1.3 Selecteren van rekengebied(en)

**Doel**  
Automatisch genereren van een rekengebied per polder op basis van ruimtelijke overlap met peilgebieden.

**Structuur**
- Hulpfuncties voor naamgeving en mappenstructuur  
- Selectiefuncties voor filteren van polders  
- Hoofdfuncties voor opbouwen van het rekengebied  

**Output**
- Voor elke polder een feature class met het rekengebied (in eigen GDB)


### 1.3 Achtergrondfunctie: selecteren van rekengebied(en)  
Verzameling van functies voor het selecteren en opbouwen van rekengebieden op basis van afvoergebieden en peilgebieden, inclusief naamgeving en outputstructuur.

In [5]:
def maak_veilige_naam(
    naam,
    *,
    target="gdb_object",   # "gdb_object" of "filesystem"
    workspace=None,
    max_len=70
):
    """
    Maakt een veilige naam voor:
    - target="filesystem"  → mappen + .gdb namen
    - target="gdb_object"  → feature classes / tabellen in een GDB

    Zet een tekst om naar een veilige naam voor gebruik in het
    bestandssysteem of binnen een geodatabase.
    Bewerkingen:
        - omzetting naar kleine letters
        - spaties vervangen door underscores
        - ongeldige tekens verwijderen/vervangen
        - voorkomen dat namen met een cijfer beginnen
        - validatie van GDB-objectnamen via arcpy
    """

    s = str(naam).strip().lower()

    # Uniforme normalisatie
    s = s.replace(" ", "_")
    s = re.sub(r"[^\w]", "_", s)   # ook - / \ etc.
    s = re.sub(r"_+", "_", s)

    # Niet beginnen met cijfer
    if s and s[0].isdigit():
        s = f"p_{s}"

    if target == "filesystem":
        return s.strip("_")

    if target == "gdb_object":
        if workspace is None:
            raise ValueError("workspace is verplicht bij target='gdb_object'")

        s = s[:max_len]
        s = s.strip("_")
        return arcpy.ValidateTableName(s, workspace)

    raise ValueError(f"Onbekend target: {target}")

def selecteer_waterschappen(
    df_polders,
    waterschappen=None
):
    """
    Filtert df_polders op opgegeven waterschappen.

    Parameters
    ----------
    df_polders : pd.DataFrame
    waterschappen : list[str] of None
        Bijvoorbeeld ["AGV", "HDSR"]
        None = alle waterschappen

    Returns
    -------
    pd.DataFrame
    """

    if waterschappen is None:
        print("--- Alle waterschappen worden meegenomen ---")
        return df_polders.copy()

    df_sel = df_polders[df_polders["Waterschap"].isin(waterschappen)].copy()

    gevonden = sorted(df_sel["Waterschap"].unique())
    print(f"--- Geselecteerde waterschappen: {gevonden} ---")

    if df_sel.empty:
        raise ValueError("--- Geen polders gevonden voor opgegeven waterschappen ---")

    return df_sel

def maak_polder_structuur(
    base_folder,
    waterschap,
    polder
):
    """
    Maakt mappenstructuur:
    Maakt indien nodig de structuur:
    base_folder/
        └── waterschap/
            └── polder/
                └── polder.gdb
    Bestaat de structuur al, dan wordt deze hergebruikt.
    """

    # veilige namen (bestandssysteem)
    ws_map = maak_veilige_naam(waterschap, target="filesystem")
    polder_map = maak_veilige_naam(polder, target="filesystem")

    ws_pad = os.path.join(base_folder, ws_map)
    polder_pad = os.path.join(ws_pad, polder_map)
    gdb_pad = os.path.join(polder_pad, f"{polder_map}.gdb")

    os.makedirs(polder_pad, exist_ok=True)

    if not arcpy.Exists(gdb_pad):
        arcpy.management.CreateFileGDB(
            out_folder_path=polder_pad,
            out_name=f"{polder_map}.gdb"
        )

        print(f"--- Aangemaakt: {gdb_pad} ---")

    return gdb_pad

def maak_rekengebied_voor_polder(
    polder_naam,
    df_polders,
    peilgebieden_fc,      # = pg_polygon (ALLE peilvakken)
    afvoergebieden_fc,
    base_output_folder,
    use_existing=True # is rekengebied al reeds aangemaakt en wil je hergebruiken, 
                        # zet use_existing=True anders use_existing=False 
):
    """
    Rekengebied:
    - rekengebied wordt aangemaakt o.b.v. shapes die in afvoergebiedenshapefile staan
    - standaard: alle peilvakken die RUIMTELIJK overlappen met de polder,
      geclipt op het afvoergebied (peilvakken blijven afzonderlijk)
    - fallback: als er GEEN overlappende peilvakken zijn → volledige polder
    
    Werkwijze
    ---------
    1. Zoek de polder op in df_polders.
    2. Maak of hergebruik de polder-geodatabase.
    3. Selecteer het afvoergebied op basis van WS_ID.
    4. Selecteer alle peilgebieden die het afvoergebied raken.
    5. Clip de geselecteerde peilgebieden op het afvoergebied.
    6. Als geen peilgebieden worden gevonden, gebruik het volledige afvoergebied als rekengebied.
    """

    print(f"--- Verwerken polder: {polder_naam} ---")

    # --- 1. Polder ophalen uit dataframe ---
    rij = df_polders.loc[df_polders["Polder"] == polder_naam]
    if rij.empty:
        raise ValueError(f"Polder '{polder_naam}' niet gevonden in dataframe")

    waterschap = rij["Waterschap"].iloc[0]
    ws_id = rij["WS_ID"].iloc[0]

    # --- 2. Outputstructuur ---
    out_gdb = maak_polder_structuur(
        base_folder=base_output_folder,
        waterschap=waterschap,
        polder=polder_naam
    )

    out_name = maak_veilige_naam(
        f"rekengebied_{polder_naam}",
        target="gdb_object",
        workspace=out_gdb
    )
    
    out_fc = os.path.join(out_gdb, out_name)

    if arcpy.Exists(out_fc):
        if use_existing:
            print(f"--- Bestaand rekengebied gebruikt voor '{polder_naam}' ---")
            return out_fc
        else:
            arcpy.management.Delete(out_fc)

    # --- 3. Afvoergebied (polder) als layer ---
    polder_lyr = "polder_lyr"
    where_polder = f"WS_ID = {repr(ws_id)}"

    arcpy.management.MakeFeatureLayer(
        afvoergebieden_fc,
        polder_lyr,
        where_polder
    )
    
    if int(arcpy.management.GetCount(polder_lyr)[0]) == 0:
        raise ValueError(f"--- Geen afvoergebied gevonden voor '{polder_naam}' ---")

    # --- 4. Peilvakken selecteren op WS_ID ---
    pg_lyr = "pg_lyr"

    where_pg = f"WS_ID = {repr(ws_id)}"

    arcpy.management.MakeFeatureLayer(
        peilgebieden_fc,
        pg_lyr,
        where_pg
    )    

    # Optioneel: extra ruimtelijke check
    arcpy.management.SelectLayerByLocation(
        in_layer=pg_lyr,
        overlap_type="INTERSECT",
        select_features=polder_lyr,
        selection_type="SUBSET_SELECTION"
    )

    pg_sel_count = int(arcpy.management.GetCount(pg_lyr)[0])

    # --- 5A. Geen peilvakken → volledige polder ---
    if pg_sel_count == 0:
        print(
            f"--- {polder_naam}: geen overlappende peilvakken "
            f"→ volledige polder gebruikt als rekengebied ---"
        )
        arcpy.management.CopyFeatures(polder_lyr, out_fc)
        return out_fc

    # --- 5B. Wel peilvakken → clippen op polder ---
    arcpy.analysis.Clip(
        in_features=pg_lyr,
        clip_features=polder_lyr,
        out_feature_class=out_fc
    )
    
    out_count = int(arcpy.management.GetCount(out_fc)[0])

    # Veiligheidscheck (zou praktisch niet meer moeten voorkomen)
    if out_count == 0:
        print(
            f"--- {polder_naam}: peilvakken geselecteerd, "
            f"maar geen geometrisch resultaat → fallback polder ---"
        )
        arcpy.management.CopyFeatures(polder_lyr, out_fc)
        return out_fc

    print(f"--- {polder_naam}: {out_count} peilvak-delen geclipt op polder ---")
    return out_fc

def maak_rekengebieden_voor_polders(
    df_polders,
    peilgebieden_fc,
    afvoergebieden_fc,
    base_output_folder,
    selectie_polders=None,
    use_existing=True
):
    """
    Maakt rekengebieden voor:
    - alle polders (selectie_polders=None)
    - of een opgegeven selectie polders
    """

    if selectie_polders is None:
        polders = df_polders["Polder"].unique().tolist()
        print(f"--- Geen selectie opgegeven → {len(polders)} polders worden verwerkt ---")
    else:
        polders = selectie_polders
        print(f"--- Selectie polders → {len(polders)} polders ---")

    resultaten = {}

    for polder in polders:
        print(f"\n--- Start polder: {polder} ---")

        try:
            out_fc = maak_rekengebied_voor_polder(
                polder_naam=polder,
                df_polders=df_polders,
                peilgebieden_fc=peilgebieden_fc,
                afvoergebieden_fc=afvoergebieden_fc,
                base_output_folder=base_output_folder,
                use_existing=use_existing)

            resultaten[polder] = out_fc

        except Exception as e:
            print(f"--- Fout bij polder '{polder}': {e} ---")

    print("\n--- Klaar met aanmaken rekengebieden ---")
    return resultaten


### 1.3 Invoer: selectie en aanmaak rekengebieden


**Doel**  
Genereren van rekengebieden voor één of meerdere polders op basis van de geselecteerde waterschappen.

**Werking**
- Selecteren van relevante polders op basis van waterschap  
- Aanmaken van outputmappen en geodatabases  
- Bepalen van rekengebied per polder (ruimtelijke selectie en clip)  

**Configuratie**
- **base_output_folder**: locatie waar resultaten worden opgeslagen  
- **geselecteerde_waterschappen**: bepaalt welke polders worden meegenomen  
- **use_existing**: hergebruik bestaande resultaten (True/False)  

**Output**
- Per polder een feature class met het rekengebied  
- Dictionary met paden naar outputbestanden  


In [6]:
# Outputlocatie voor resultaten
base_output_folder = r"D:\04_results\27072026"

# Selectie van waterschappen
geselecteerde_waterschappen = ["AGV"]

# Filter polders op waterschap
df_polders_sel = selecteer_waterschappen(
    df_polders=df_rvw,
    waterschappen=geselecteerde_waterschappen
)

# Aanmaken rekengebieden
resultaten = maak_rekengebieden_voor_polders(
    df_polders=df_polders_sel,
    peilgebieden_fc=pg_polygon,     
    afvoergebieden_fc=afvoer_geb,
    base_output_folder=base_output_folder,
    selectie_polders=None,
    use_existing=True
)

--- Geselecteerde waterschappen: ['AGV'] ---
--- Geen selectie opgegeven → 154 polders worden verwerkt ---

--- Start polder: Polder Holland en Sticht west ---
--- Verwerken polder: Polder Holland en Sticht west ---
--- Bestaand rekengebied gebruikt voor 'Polder Holland en Sticht west' ---

--- Start polder: Westerpark ---
--- Verwerken polder: Westerpark ---
--- Bestaand rekengebied gebruikt voor 'Westerpark' ---

--- Start polder: Gansenhoef west ---
--- Verwerken polder: Gansenhoef west ---
--- Bestaand rekengebied gebruikt voor 'Gansenhoef west' ---

--- Start polder: Breukelen boezempeil ---
--- Verwerken polder: Breukelen boezempeil ---
--- Bestaand rekengebied gebruikt voor 'Breukelen boezempeil' ---

--- Start polder: De Lange Bretten ---
--- Verwerken polder: De Lange Bretten ---
--- Bestaand rekengebied gebruikt voor 'De Lange Bretten' ---

--- Start polder: Stadsboezem Amsterdam ---
--- Verwerken polder: Stadsboezem Amsterdam ---
--- Bestaand rekengebied gebruikt voor 'Stads

--- Start polder: Baambrugge Oostzijds ---
--- Verwerken polder: Baambrugge Oostzijds ---
--- Bestaand rekengebied gebruikt voor 'Baambrugge Oostzijds' ---

--- Start polder: Nieuwe Keverdijksche Polder en Hilversumse Bovenme ---
--- Verwerken polder: Nieuwe Keverdijksche Polder en Hilversumse Bovenme ---
--- Bestaand rekengebied gebruikt voor 'Nieuwe Keverdijksche Polder en Hilversumse Bovenme' ---

--- Start polder: Zuidpolder beoosten Muiden ---
--- Verwerken polder: Zuidpolder beoosten Muiden ---
--- Bestaand rekengebied gebruikt voor 'Zuidpolder beoosten Muiden' ---

--- Start polder: Bloemendalerpolder en Gemeenschapspolder Oost ---
--- Verwerken polder: Bloemendalerpolder en Gemeenschapspolder Oost ---
--- Bestaand rekengebied gebruikt voor 'Bloemendalerpolder en Gemeenschapspolder Oost' ---

--- Start polder: Oosterpark ---
--- Verwerken polder: Oosterpark ---
--- Bestaand rekengebied gebruikt voor 'Oosterpark' ---

--- Start polder: Bijlmer ---
--- Verwerken polder: Bijlmer --

# Deel 2 - Koppel watervlakken met peil rekengebied

### 2.1 Watervlakken met corresponderend peil + buffer voor oever ###

**Doel**  
Koppelen van peilinformatie aan watervlakken en genereren van een buffer rondom watergangen voor analyse van de oeverzone.

**Werking**
- Clipping van watervlakken op het rekengebied  
- Koppelen van peilen via ruimtelijke join met peilgebieden  
- Controleren op aanwezigheid van benodigde peilvelden  
- Genereren van een buffer rondom watervlakken  

**Output**
- Geclipte watervlakken binnen het rekengebied  
- Watervlakken met gekoppelde peilinformatie  
- Buffer rondom watervlakken (oeverzone)  
- Dictionary met paden naar gegenereerde datasets  



### 2.1 Achtergrondfunctie: watervlakken met corresponderend peil + buffer voor oever

Deze functie verwerkt per polder de watervlakken door deze te clippen op het rekengebied, te verrijken met peilinformatie via een spatial join en een buffer te genereren voor de oeverzone. 
De functie is idempotent (hergebruik van bestaande resultaten) en zorgt voor consistente datasetnamen en opslag per polder.

In [7]:
def update_water_levels_per_polder(
    rekengebieden_dict,
    peilvakken_fc,
    water_areas,
    level_fields=("Peil_wi", "Peil_zo"),
    buffer_meter=4,
    use_existing=True
):
    """
    Koppelt waterpeilen aan watervlakken binnen het rekengebied van
    iedere polder.

    Werkwijze
    ---------
    1. Clip waterobjecten op het rekengebied.
    2. Koppel peilinformatie vanuit de peilvakken via een spatial join.
    3. Controleer of de vereiste peilvelden aanwezig zijn.
    4. Maak een buffer rond de waterobjecten.
    5. Sla alle tussenresultaten op in een tijdelijke geodatabase
       per polder.

    Bestaande resultaten kunnen worden hergebruikt met
    use_existing=True.

    Parameters
    ----------
    rekengebieden_dict : dict[str, str]
        Dictionary met per polder het pad naar het rekengebied. 
        Deze dictionary kan in de rest van het script hieronder gebruikt worden voor verdere berekeningen.  

    peilvakken_fc : str
        Feature class met peilgebieden inclusief peilvelden.

    water_areas : str
        Polygonlaag met wateroppervlakken.

    level_fields : tuple[str]
        Velden in attribute table met winter- en zomerpeil.

    buffer_meter : default=4
        Bufferafstand rondom waterobjecten.

    use_existing : bool, default=True
        Hergebruik bestaande tussenresultaten indien aanwezig.

    Returns
    -------
    dict
    Per polder:
    {
        "Poldernaam": {
            "temp_gdb": "...",
            "clip": "...",
            "join": "...",
            "buffer": "..."
        }
    }
    """

    arcpy.env.overwriteOutput = False
    resultaten = {}

    for polder, rekengebied_fc in rekengebieden_dict.items():
        print(f"\n➡️ Peilen koppelen voor polder: {polder}")

        # --------------------------------------------------
        # Veilige poldernaam & paden
        # --------------------------------------------------
        safe_fs = maak_veilige_naam(
            polder,
            target="filesystem"
        )

        polder_gdb = arcpy.Describe(rekengebied_fc).path
        polder_map = os.path.dirname(polder_gdb)

        safe_gdb = maak_veilige_naam(
            polder,
            target="gdb_object",
            workspace=polder_gdb
        )

        temp_gdb = os.path.join(
            polder_map, f"_temp_{safe_fs}.gdb"
        )
        if not arcpy.Exists(temp_gdb):
            arcpy.management.CreateFileGDB(
                polder_map, f"_temp_{safe_fs}.gdb"
            )
            
        # --------------------------------------------------
        # Consistente dataset-namen
        # --------------------------------------------------
        clipped_water = os.path.join(
            temp_gdb, f"water_clipped_{safe_gdb}"
        )
        sj_water = os.path.join(
            temp_gdb, f"sj_water_{safe_gdb}"
        )
        water_buffer_fc = os.path.join(
            temp_gdb,
            f"water_buffer_{safe_gdb}_{buffer_meter}m"
        )
        

        if not use_existing:
            for fc in (clipped_water, sj_water, water_buffer_fc):
                if arcpy.Exists(fc):
                    arcpy.management.Delete(fc)

        # --------------------------------------------------
        # Hergebruik bestaande data
        # --------------------------------------------------
        if use_existing and all(
            arcpy.Exists(p)
            for p in (clipped_water, sj_water, water_buffer_fc)
        ):
            print("♻️  Bestaande resultaten gebruikt")
            resultaten[polder] = {
                "temp_gdb": temp_gdb,
                "clip": clipped_water,
                "join": sj_water,
                "buffer": water_buffer_fc
            }
            continue

        rg_lyr = f"rg_lyr_{safe_gdb}" 

        try:
            # --------------------------------------------------
            # Rekengebied layer
            # --------------------------------------------------
            arcpy.management.MakeFeatureLayer(
                rekengebied_fc,
                rg_lyr
            )

            # --------------------------------------------------
            # 1. Water clippen (ÉÉN KEER)
            # --------------------------------------------------
            if not arcpy.Exists(clipped_water):
                arcpy.analysis.Clip(
                    in_features=water_areas,
                    clip_features=rg_lyr,
                    out_feature_class=clipped_water
                )

            if int(arcpy.management.GetCount(clipped_water)[0]) == 0:
                print("⚠️ Geen waterobjecten → polder overgeslagen")
                continue

            # --------------------------------------------------
            # 2. Spatial join met peilvakken
            # --------------------------------------------------
            if not arcpy.Exists(sj_water):
                arcpy.analysis.SpatialJoin(
                    target_features=clipped_water,
                    join_features=peilvakken_fc,
                    out_feature_class=sj_water,
                    join_operation="JOIN_ONE_TO_ONE",
                    match_option="LARGEST_OVERLAP"
                )

            aanwezige_velden = {f.name for f in arcpy.ListFields(sj_water)}
            ontbrekend = set(level_fields) - aanwezige_velden
            if ontbrekend:
                raise RuntimeError(
                    f"Ontbrekende peilvelden in sj_water: {ontbrekend}"
                )

            # --------------------------------------------------
            # 3. Buffer (ÉÉN KEER, consistente naam)
            # --------------------------------------------------
            if not arcpy.Exists(water_buffer_fc):
                arcpy.analysis.Buffer(
                    in_features=sj_water,
                    out_feature_class=water_buffer_fc,
                    buffer_distance_or_field=f"{buffer_meter} Meters",
                    dissolve_option="NONE"
                )

            resultaten[polder] = {
                "temp_gdb": temp_gdb,
                "clip": clipped_water,
                "join": sj_water,
                "buffer": water_buffer_fc
            }

            print("✅ Peilen gekoppeld en buffer aangemaakt")

        finally:
            # --------------------------------------------------
            # Opruimen layers / locks
            # --------------------------------------------------
            if arcpy.Exists(rg_lyr):
                arcpy.management.Delete(rg_lyr)
            gc.collect()

    print("\n✅ Klaar met koppelen van peilen per polder")
    return resultaten



### 2.1 Invoer watervlakken met corresponderend peil + buffer voor oever

**Doel**  
Uitvoeren van de bewerking waarbij watervlakken worden gekoppeld aan peilinformatie en voorzien van een buffer voor analyse van de oeverzone.

**Configuratie**
- **rekengebieden_dict**: dictionary met rekengebieden per polder  
- **peilvakken_fc**: feature class met peilgebieden en peilen  
- **water_areas**: watervlakken (watergangen)  
- **level_fields**: velden met zomer- en winterpeil  
- **buffer_meter**: bufferafstand rondom watervlakken (in meters)  
- **use_existing**: hergebruik bestaande resultaten (True/False)  

**Output**
- Dictionary met per polder:
  - geclipte watervlakken  
  - watervlakken met gekoppelde peilen  
  - buffer rondom watervlakken  


In [8]:

    # ==========================================================
    # Waterpeilen koppelen aan watervlakken
    # ==========================================================
    #
    # Per polder:
    # - watervlakken clippen op het rekengebied
    # - peilinformatie uit peilvakken koppelen
    # - buffer rond watervlakken maken
    # - tussenresultaten opslaan in tijdelijke GDB
    #
    # Resultaat:
    # dictionary met paden naar clip-, join- en bufferdatasets

wv_peil = update_water_levels_per_polder(
    rekengebieden_dict=resultaten,
    peilvakken_fc=pg_polygon,
    water_areas=water_vlakken,
    level_fields=("Peil_wi", "Peil_zo"),
    buffer_meter=4,
    use_existing=False)


➡️ Peilen koppelen voor polder: Polder Holland en Sticht west
✅ Peilen gekoppeld en buffer aangemaakt

➡️ Peilen koppelen voor polder: Westerpark
✅ Peilen gekoppeld en buffer aangemaakt

➡️ Peilen koppelen voor polder: Gansenhoef west
✅ Peilen gekoppeld en buffer aangemaakt

➡️ Peilen koppelen voor polder: Breukelen boezempeil
✅ Peilen gekoppeld en buffer aangemaakt

➡️ Peilen koppelen voor polder: De Lange Bretten
✅ Peilen gekoppeld en buffer aangemaakt

➡️ Peilen koppelen voor polder: Stadsboezem Amsterdam
✅ Peilen gekoppeld en buffer aangemaakt

➡️ Peilen koppelen voor polder: Noordpolder beoosten Muiden
✅ Peilen gekoppeld en buffer aangemaakt

➡️ Peilen koppelen voor polder: Riekerpolder
✅ Peilen gekoppeld en buffer aangemaakt

➡️ Peilen koppelen voor polder: Veldhuiswetering
✅ Peilen gekoppeld en buffer aangemaakt

➡️ Peilen koppelen voor polder: Horn- en Kuyerpolder
✅ Peilen gekoppeld en buffer aangemaakt

➡️ Peilen koppelen voor polder: Aetsveldse Polder Oost
✅ Peilen gekoppeld

### 2.2 Rasteriseer watervlakken met corresponderend waterpeil 

**Doel**  
Omzetten van watervlakken met peilinformatie naar rasterdata, zodat deze gebruikt kunnen worden in verdere ruimtelijke berekeningen.

**Werking**
- Selecteren van watervlakken met gekoppelde peilinformatie per polder  
- Omzetten van polygonen naar raster (cell center methode)  
- Schalen van peilwaarden met een vaste factor (int_mult)  
- Opslaan van raster als integer voor consistente verwerking  

**Output**
- Per polder een rasterbestand met waterpeilen  
- Dictionary met paden naar de gegenereerde rasters  


### 2.2 Achtergrondfunctie: Rasteriseer watervlakken met corresponderend waterpeil

Deze functie zet watervlakken om naar rasterformaat waarbij elke cel een peilwaarde krijgt op basis van het middenpunt van de cel.  
De peilwaarden worden geschaald en opgeslagen als integer raster om nauwkeurige en efficiënte verdere berekeningen mogelijk te maken.

In [9]:
def rasterize_waterpeilen(
    peil_resultaten_per_polder,
    peil_field,          # bijv. "Peil_zo", "Peil_wi"
    cell_size,
    int_mult=1000,
    use_existing=True
):
    """
    Rasteriseert een waterpeilveld per polder.

    Op basis van de door update_water_levels_per_polder()
    aangemaakte waterobjecten met gekoppelde peilen wordt
    voor iedere polder een raster gemaakt.

    Werkwijze
    ---------
    1. Controleer of het opgegeven peilveld aanwezig is.
    2. Controleer of het veld waarden bevat.
    3. Rasteriseer de waterobjecten met PolygonToRaster.
    4. Schaal de rasterwaarden met int_mult.
    5. Converteer naar een integer raster.
    6. Sla het definitieve raster op in de tijdelijke GDB
       van de polder.

    De rasterwaarde vertegenwoordigt het peil van het
    watervlak waarin het middelpunt van de rastercel valt
    (CELL_CENTER).

    Parameters
    ----------
    peil_resultaten_per_polder : dict
        Output van update_water_levels_per_polder().

    peil_field : str
        Naam van het te rasteriseren peilveld,
        bijvoorbeeld 'Peil_zo' of 'Peil_wi'.

    cell_size : float
        Rastercelgrootte in meters.

    int_mult : int, default=1000
        Schaalfactor waarmee float-peilen worden omgezet
        naar gehele getallen.

    use_existing : bool, default=True
        Hergebruik reeds aanwezige rasters.

    Returns
    -------
    dict[str, str]

    Dictionary met per polder het pad naar het
    aangemaakte raster.

    Voorbeeld
    ---------
    {
        "Polder_A": "...\\_temp_polder_a.gdb\\Peil_zo",
        "Polder_B": "...\\_temp_polder_b.gdb\\Peil_zo"
    }
    """
#     from arcpy.sa import Times, Int

    if not isinstance(peil_field, str) or not peil_field:
        raise ValueError("peil_field moet een geldige veldnaam (string) zijn")

    print(f"\n--- Start rasteriseren peilveld: {peil_field} ---")

    arcpy.env.overwriteOutput = True
    arcpy.CheckOutExtension("Spatial")

    resultaten = {}

    for polder, info in peil_resultaten_per_polder.items():

        water_fc = info["join"]          # sj_water_<safe_polder>
        temp_gdb = info["temp_gdb"]      # _temp_<safe_polder>.gdb

        print(f"\n--- Rasteriseren '{peil_field}' voor polder: {polder} ---")

        # --------------------------------------------------
        # Output raster (CONSISTENT)
        # --------------------------------------------------
        out_raster = os.path.join(temp_gdb, peil_field)

        # ✅ Bestaand raster hergebruiken
        if use_existing and arcpy.Exists(out_raster):
            print(f"--- Bestaand raster gebruikt: {out_raster} ---")
            resultaten[polder] = out_raster
            continue

        # --------------------------------------------------
        # Controle peilveld
        # --------------------------------------------------
        veldnamen = [f.name for f in arcpy.ListFields(water_fc)]
        if peil_field not in veldnamen:
            raise ValueError(
                f"Veld '{peil_field}' bestaat niet in {water_fc}"
            )

        with arcpy.da.SearchCursor(water_fc, [peil_field]) as cur:
            if not any(row[0] is not None for row in cur):
                print(
                    f"--- Geen waarden in veld '{peil_field}', "
                    f"polder wordt overgeslagen ---"
                )
                continue

        # --------------------------------------------------
        # Tijdelijke rasterpaden
        # --------------------------------------------------
        tmp_float = os.path.join(
            temp_gdb, f"_tmp_{peil_field}_float"
        )
        tmp_scaled = os.path.join(
            temp_gdb, f"_tmp_{peil_field}_x{int_mult}"
        )

        for r in (tmp_float, tmp_scaled, out_raster):
            if arcpy.Exists(r):
                arcpy.management.Delete(r)

        # --------------------------------------------------
        # Polygon → Raster (FLOAT)
        # --------------------------------------------------
        arcpy.conversion.PolygonToRaster(
            in_features=water_fc,
            value_field=peil_field,
            out_rasterdataset=tmp_float,
            cell_assignment="CELL_CENTER",   
            cellsize=cell_size
        )

        # --------------------------------------------------
        # Schalen: * int_mult
        # --------------------------------------------------
        Times(tmp_float, int_mult).save(tmp_scaled)

        # --------------------------------------------------
        # INTEGER raster (32-bit signed)
        # --------------------------------------------------
        int_raster = Int(tmp_scaled)

        arcpy.management.CopyRaster(
            in_raster=int_raster,
            out_rasterdataset=out_raster,
            pixel_type="32_BIT_SIGNED"
        )

        # --------------------------------------------------
        # Cleanup tijdelijke rasters
        # --------------------------------------------------
        for r in (tmp_float, tmp_scaled):
            if arcpy.Exists(r):
                arcpy.management.Delete(r)

        print(f"--- Raster aangemaakt (INT x{int_mult}): {out_raster} ---")
        resultaten[polder] = out_raster

    arcpy.CheckInExtension("Spatial")
    print("\n--- Klaar met rasteriseren waterpeilen ---")

    return resultaten


### 2.2 Invoer: Rasteriseer watervlakken met corresponderend waterpeil


**Doel**  
Uitvoeren van de rasterisatie van watervlakken met gekoppelde peilinformatie voor een gekozen peiltype.

**Configuratie**
- **peil_resultaten_per_polder**: dictionary met watervlakken en peilinformatie per polder  
- **peil_field**: te rasteriseren peilveld (bijv. 'Peil_zo' of 'Peil_wi')  
- **cell_size**: resolutie van het raster (in meters)  
- **use_existing**: hergebruik bestaande rasters (True/False)  

**Output**
- Per polder een rasterbestand met waterpeilen  
- Dictionary met paden naar de gegenereerde rasters  


In [10]:
# ==========================================================
# Rasterisatie zomerpeilen
# ==========================================================
#
# Maakt per polder een raster met zomerpeilen (Peil_zo).
# Rasterresolutie: 0,5 meter.
# use_existing=False zorgt ervoor dat eventuele bestaande
# rasters opnieuw worden berekend.
#


wv_zo = rasterize_waterpeilen(
    peil_resultaten_per_polder=wv_peil,
    peil_field='Peil_zo',
    cell_size=0.5,
    use_existing=False
)


--- Start rasteriseren peilveld: Peil_zo ---

--- Rasteriseren 'Peil_zo' voor polder: Polder Holland en Sticht west ---
--- Raster aangemaakt (INT x1000): D:\04_results\27072026\agv\polder_holland_en_sticht_west\_temp_polder_holland_en_sticht_west.gdb\Peil_zo ---

--- Rasteriseren 'Peil_zo' voor polder: Westerpark ---
--- Raster aangemaakt (INT x1000): D:\04_results\27072026\agv\westerpark\_temp_westerpark.gdb\Peil_zo ---

--- Rasteriseren 'Peil_zo' voor polder: Gansenhoef west ---
--- Raster aangemaakt (INT x1000): D:\04_results\27072026\agv\gansenhoef_west\_temp_gansenhoef_west.gdb\Peil_zo ---

--- Rasteriseren 'Peil_zo' voor polder: Breukelen boezempeil ---
--- Raster aangemaakt (INT x1000): D:\04_results\27072026\agv\breukelen_boezempeil\_temp_breukelen_boezempeil.gdb\Peil_zo ---

--- Rasteriseren 'Peil_zo' voor polder: De Lange Bretten ---
--- Raster aangemaakt (INT x1000): D:\04_results\27072026\agv\de_lange_bretten\_temp_de_lange_bretten.gdb\Peil_zo ---

--- Rasteriseren 'Peil_

--- Raster aangemaakt (INT x1000): D:\04_results\27072026\agv\zuid_bijlmer\_temp_zuid_bijlmer.gdb\Peil_zo ---

--- Rasteriseren 'Peil_zo' voor polder: Holendrechter- en Bullewijker Polder (zuid en west ---
--- Raster aangemaakt (INT x1000): D:\04_results\27072026\agv\holendrechter_en_bullewijker_polder_zuid_en_west\_temp_holendrechter_en_bullewijker_polder_zuid_en_west.gdb\Peil_zo ---

--- Rasteriseren 'Peil_zo' voor polder: Polder de Nieuwe Bullewijk en Holendrechter- en Bu ---
--- Raster aangemaakt (INT x1000): D:\04_results\27072026\agv\polder_de_nieuwe_bullewijk_en_holendrechter_en_bu\_temp_polder_de_nieuwe_bullewijk_en_holendrechter_en_bu.gdb\Peil_zo ---

--- Rasteriseren 'Peil_zo' voor polder: Gemeenschapspolder West (Tuincomplex Linnaeus) ---
--- Raster aangemaakt (INT x1000): D:\04_results\27072026\agv\gemeenschapspolder_west_tuincomplex_linnaeus\_temp_gemeenschapspolder_west_tuincomplex_linnaeus.gdb\Peil_zo ---

--- Rasteriseren 'Peil_zo' voor polder: Gemeenschapspolder West -

# Deel 3 - Voorbewerking AHN rekengebied

### 3.1 Merge AHN grondfilter en AHN Lizard

**Doel**  
Genereren van een consistente AHN-raster per polder door basisdata en Lizard-AHN te combineren, waarbij waterzones apart worden behandeld.

**Werking**
- Clippen van AHN-basisdata en Lizard-AHN op het rekengebied  
- Selecteren van waterzones en omzetten naar raster (no-data waarde)  
- Maskeren van water in Lizard-AHN met vaste waarde  
- Combineren van Lizard-AHN met basisdata in waterzones  
- Opslaan van gecombineerd AHN per polder  

**Output**
- Per polder een AHN-raster met consistente dekking  
- Tijdelijke rasters voor tussenstappen (in temp GDB)  


### 3.1 Achtergrondfunctie: merge AHN grondfilter en AHN Lizard

Deze functie combineert twee AHN-bronnen:
- Lizard-AHN voor hoge resolutie  
- Basisdata-AHN als aanvulling in waterzones  

Watergebieden worden expliciet gemaskeerd en vervangen door basisdata om een robuuste en volledige hoogtekaart te verkrijgen per polder.

In [11]:
def process_ahn_for_all_polders(
    rekengebieden_dict,
    ahn_basis,
    ahn_lizard,
    buffer_meter=4,
    cleanup=True
):
    
    """
    Combineert AHN-basis en AHN-Lizard per polder tot één definitief
    AHN-raster.

    Werkwijze
    ---------
    1. Maak een buffer rond de watervlakken.
    2. Clip de buffer op het rekengebied.
    3. Rasteriseer de waterbuffer.
    4. Clip AHN-basis op de polder.
    5. Clip AHN-basis op de waterbuffer.
    6. Clip AHN-Lizard op de polder.
    7. Vervang waterzones in AHN-Lizard door NoData.
    8. Vul de waterzones met waarden uit AHN-basis.
    9. Sla het gecombineerde raster op als ahn_final_<polder>.

    Hierdoor wordt AHN-Lizard gebruikt als primaire bron en
    AHN-basis binnen watergebieden.

    Parameters
    ----------
    rekengebieden_dict : dict[str, str]
        Dictionary met per polder het pad naar het rekengebied.

    ahn_basis : str
        Raster met AHN-basisgegevens.

    ahn_lizard : str
        Raster met AHN-Lizardgegevens.

    buffer_meter : int, default=4
        Bufferafstand rondom watervlakken.

    cleanup : bool, default=True
        Verwijder tijdelijke rasters na afronding.

    Resultaat
    ----------
    Per polder wordt een raster opgeslagen:

    ahn_final_<polder>

    in de polder-geodatabase.
    """

    from arcpy.sa import Con, SetNull, FocalStatistics, EqualTo, Times

    arcpy.env.overwriteOutput = True
    WATER_VALUE = -999999

    for polder, rekengebied_fc in rekengebieden_dict.items():

        print(f"\n=== Start AHN verwerking: {polder} ===")

        safe_fs = maak_veilige_naam(
            polder,
            target="filesystem"
        )

        safe_gdb = maak_veilige_naam(
            polder,
            target="gdb_object",
            workspace=arcpy.Describe(rekengebied_fc).path
        )

        # ----------------------------------------------------------
        # Paden
        # ----------------------------------------------------------
        polder_gdb = arcpy.Describe(rekengebied_fc).path
        
        final = os.path.join(
            polder_gdb, f"ahn_final_{safe_gdb}"
        )

        # Sla AHN over als deze al bestaat
        if arcpy.Exists(final):
            print(
                f"--- AHN bestaat al voor polder {polder} "
                "→ verwerking overgeslagen ---"
            )
            continue        
        
        polder_map = os.path.dirname(polder_gdb)

        temp_gdb = os.path.join(polder_map, f"_temp_{safe_fs}.gdb")
        if not arcpy.Exists(temp_gdb):
            arcpy.management.CreateFileGDB(
                polder_map, f"_temp_{safe_fs}.gdb"
            )

        # ----------------------------------------------------------
        # WATER → buffer → clip op polder
        # ----------------------------------------------------------
        water_clip = os.path.join(
            temp_gdb, f"water_clipped_{safe_gdb}"
        )
        if not arcpy.Exists(water_clip):
            raise RuntimeError(
                f"water_clipped ontbreekt voor {polder}"
            )

        water_buf = os.path.join(
            temp_gdb, f"water_buffer_{buffer_meter}m"
        )
        arcpy.analysis.Buffer(
            water_clip,
            water_buf,
            f"{buffer_meter} Meters",
            dissolve_option="ALL"
        )

        water_buf_polder = os.path.join(
            temp_gdb,
            f"water_buffer_{safe_gdb}_{buffer_meter}m"
        )
      
        arcpy.management.RepairGeometry(water_buf)
        arcpy.management.RepairGeometry(rekengebied_fc)
         

        arcpy.analysis.Clip(
            water_buf,
            rekengebied_fc,
            water_buf_polder
        )

        # Count bepalen ná clip
        count = int(arcpy.management.GetCount(water_buf_polder)[0])

        if count == 0:
            print(
                f"--- Geen watervlak in polder {polder}; "
                "AHN-verwerking wordt overgeslagen ---"
            )
            # Niet verwijderen, alleen tijdelijke buffer!
            arcpy.management.Delete(water_buf)  
            continue

        arcpy.management.Delete(water_buf)

        arcpy.management.AddField(
            water_buf_polder, "NO_VALUE", "LONG"
        )
        arcpy.management.CalculateField(
            water_buf_polder,
            "NO_VALUE",
            str(WATER_VALUE),
            "PYTHON3"
        )

        # ----------------------------------------------------------
        # WATERBUFFER → RASTER (−999999)
        # ----------------------------------------------------------
        arcpy.env.snapRaster = ahn_lizard
        arcpy.env.cellSize = ahn_lizard

        water_raster = os.path.join(
            temp_gdb, f"water_hi_{safe_gdb}"
        )
        arcpy.conversion.PolygonToRaster(
            water_buf_polder,
            "NO_VALUE",
            water_raster,
            "MAXIMUM_AREA",
            cellsize=ahn_lizard
        )

        # ----------------------------------------------------------
        # AHN BASIS → POLDER
        # ----------------------------------------------------------
        basis_polder = os.path.join(
            temp_gdb, f"ahn_basis_polder_{safe_gdb}"
        )
        arcpy.management.Clip(
            ahn_basis,
            "#",
            basis_polder,
            rekengebied_fc,
            None,
            "ClippingGeometry"
        )

        # ----------------------------------------------------------
        # AHN BASIS → WATERBUFFER
        # ----------------------------------------------------------
        basis_water = os.path.join(
            temp_gdb, f"ahn_basis_water_{safe_gdb}"
        )
        arcpy.management.Clip(
            basis_polder,
            "#",
            basis_water,
            water_buf_polder,
            None,
            "ClippingGeometry"
        )

#         # ----------------------------------------------------------
#         # AHN BASIS WATER ×10 (als ahn basis nog eerst nog in 
#         # de juiste eenheid gezet moet worden). 
#         # ----------------------------------------------------------
#         basis_water_x10 = os.path.join(
#             temp_gdb, f"ahn_basis_water_x10_{safe_gdb}"
#         )
#         Times(basis_water, 10).save(basis_water_x10)

        # ----------------------------------------------------------
        # AHN LIZARD → POLDER
        # ----------------------------------------------------------
        lizard_polder = os.path.join(
            temp_gdb, f"ahn_lizard_polder_{safe_gdb}"
        )
        arcpy.management.Clip(
            ahn_lizard,
            "#",
            lizard_polder,
            rekengebied_fc,
            None,
            "ClippingGeometry"
        )

        # ----------------------------------------------------------
        # LIZARD → MOSAIC MET WATER (-999999)
        # ----------------------------------------------------------
        lizard_water = os.path.join(
            temp_gdb, f"ahn_lizard_water_{safe_gdb}"
        )
        arcpy.management.MosaicToNewRaster(
            [water_raster, lizard_polder],
            temp_gdb,
            os.path.basename(lizard_water),
            pixel_type="32_BIT_FLOAT",
            number_of_bands=1,
            mosaic_method="FIRST"
        )

        lizard_no_water = os.path.join(
            temp_gdb, f"ahn_lizard_no_water_{safe_gdb}"
        )
        SetNull(
            EqualTo(lizard_water, WATER_VALUE),
            lizard_water
        ).save(lizard_no_water)

        # ----------------------------------------------------------
        # MOSAIC MET GESCHAALDE BASIS (×10)
        # ----------------------------------------------------------
        combined = os.path.join(
            temp_gdb, f"ahn_combined_{safe_gdb}"
        )
        arcpy.management.MosaicToNewRaster(
            [basis_water, lizard_no_water],
            temp_gdb,
            os.path.basename(combined),
            pixel_type="32_BIT_SIGNED",
            number_of_bands=1,
            mosaic_method="FIRST"
        )

        # ----------------------------------------------------------
        # OPSLAAN 
        # ----------------------------------------------------------
        final = os.path.join(
            polder_gdb, f"ahn_final_{safe_gdb}"
        )

        arcpy.management.CopyRaster(
            combined,
            final
        )

        print(f"--- AHN final opgeslagen (zonder interpolatie) → {final} ---")

        # ----------------------------------------------------------
        # CLEANUP: tijdelijke rasters + waterbuffer-polder FC
        # ----------------------------------------------------------
        if cleanup:
            print("--- Cleanup: alleen tijdelijke rasters verwijderen ---")

            tijdelijke_prefixes = (
                "water_hi_",
                "ahn_basis_",
                "ahn_basis_water_x10_",
                "ahn_lizard_",
                "ahn_combined_",
            )

            arcpy.env.workspace = temp_gdb
            for r in arcpy.ListRasters() or []:
                if r.startswith(tijdelijke_prefixes):
                    arcpy.management.Delete(r)
            arcpy.env.workspace = None

#             if arcpy.Exists(water_buf_polder):
#                 try:
#                     arcpy.management.Delete(water_buf_polder)
#                 except Exception as e:
#                     print(
#                         f"--- Kon waterbuffer niet verwijderen ---"
#                         f"({water_buf_polder}): {e}"
#                     )

    print("\n--- Alle polders correct verwerkt ---")


### 3.1 Invoer: merge AHN grondfilter en AHN Lizard

**Doel**  
Uitvoeren van de voorbewerking van AHN-data door basisdata en Lizard-AHN te combineren per polder.

**Configuratie**
- **rekengebieden_dict**: dictionary met rekengebieden per polder  
- **ahn_basis**: AHN basisdata (landelijke dekking)  
- **ahn_lizard**: AHN via Lizard (hogere resolutie)  
- **buffer_meter**: bufferafstand rondom watervlakken (in meters)  
- **cleanup**: verwijderen van tijdelijke bestanden (True/False)  

**Output**
- Per polder een gecombineerd AHN-raster  
- Tijdelijke bestanden in de bijbehorende temp GDB  



In [12]:
# ==========================================================
# AHN verwerken per polder
# ==========================================================
#
# Combineert AHN-Lizard en AHN-basisdata tot één definitief
# AHN-raster per polder. Watergebieden worden gevuld met
# waarden uit de basis-AHN.

process_ahn_for_all_polders(
    rekengebieden_dict=resultaten,
    ahn_basis=dem_basisdata,
    ahn_lizard=dem_lizard,
    buffer_meter=4,
    cleanup=True)


=== Start AHN verwerking: Polder Holland en Sticht west ===
--- AHN final opgeslagen (zonder interpolatie) → D:\04_results\27072026\agv\polder_holland_en_sticht_west\polder_holland_en_sticht_west.gdb\ahn_final_polder_holland_en_sticht_west ---
--- Cleanup: alleen tijdelijke rasters verwijderen ---

=== Start AHN verwerking: Westerpark ===
--- AHN final opgeslagen (zonder interpolatie) → D:\04_results\27072026\agv\westerpark\westerpark.gdb\ahn_final_westerpark ---
--- Cleanup: alleen tijdelijke rasters verwijderen ---

=== Start AHN verwerking: Gansenhoef west ===
--- AHN final opgeslagen (zonder interpolatie) → D:\04_results\27072026\agv\gansenhoef_west\gansenhoef_west.gdb\ahn_final_gansenhoef_west ---
--- Cleanup: alleen tijdelijke rasters verwijderen ---

=== Start AHN verwerking: Breukelen boezempeil ===
--- AHN final opgeslagen (zonder interpolatie) → D:\04_results\27072026\agv\breukelen_boezempeil\breukelen_boezempeil.gdb\ahn_final_breukelen_boezempeil ---
--- Cleanup: alleen tij

--- Cleanup: alleen tijdelijke rasters verwijderen ---

=== Start AHN verwerking: Venserpolder (volkstuinpark Amstelglorie) ===
--- AHN final opgeslagen (zonder interpolatie) → D:\04_results\27072026\agv\venserpolder_volkstuinpark_amstelglorie\venserpolder_volkstuinpark_amstelglorie.gdb\ahn_final_venserpolder_volkstuinpark_amstelglorie ---
--- Cleanup: alleen tijdelijke rasters verwijderen ---

=== Start AHN verwerking: Polder De Toekomst ===
--- AHN final opgeslagen (zonder interpolatie) → D:\04_results\27072026\agv\polder_de_toekomst\polder_de_toekomst.gdb\ahn_final_polder_de_toekomst ---
--- Cleanup: alleen tijdelijke rasters verwijderen ---

=== Start AHN verwerking: Polder Groot Wilnis Vinkeveen ===
--- AHN final opgeslagen (zonder interpolatie) → D:\04_results\27072026\agv\polder_groot_wilnis_vinkeveen\polder_groot_wilnis_vinkeveen.gdb\ahn_final_polder_groot_wilnis_vinkeveen ---
--- Cleanup: alleen tijdelijke rasters verwijderen ---

=== Start AHN verwerking: Baambrugge Westzijds

--- AHN final opgeslagen (zonder interpolatie) → D:\04_results\27072026\agv\lintbebouwing_dwarsdijk\lintbebouwing_dwarsdijk.gdb\ahn_final_lintbebouwing_dwarsdijk ---
--- Cleanup: alleen tijdelijke rasters verwijderen ---

=== Start AHN verwerking: Over-Diemen ===
--- AHN final opgeslagen (zonder interpolatie) → D:\04_results\27072026\agv\over_diemen\over_diemen.gdb\ahn_final_over_diemen ---
--- Cleanup: alleen tijdelijke rasters verwijderen ---

=== Start AHN verwerking: W.H. Vliegenbos ===
--- AHN final opgeslagen (zonder interpolatie) → D:\04_results\27072026\agv\w_h_vliegenbos\w_h_vliegenbos.gdb\ahn_final_w_h_vliegenbos ---
--- Cleanup: alleen tijdelijke rasters verwijderen ---

=== Start AHN verwerking: Over-Diemen (Zeehoeve) ===
--- AHN final opgeslagen (zonder interpolatie) → D:\04_results\27072026\agv\over_diemen_zeehoeve\over_diemen_zeehoeve.gdb\ahn_final_over_diemen_zeehoeve ---
--- Cleanup: alleen tijdelijke rasters verwijderen ---

=== Start AHN verwerking: Over-Diemen (noor

### 3.2 Merge resulterende AHN met peil raster

**Doel**  
Combineren van AHN en peilrasters per polder tot één volledig rasterbestand voor verdere analyse.

**Werking**
- Clippen van AHN-raster op het rekengebied  
- Combineren van AHN met peilraster via mosaic  
- Interpoleren van ontbrekende waarden  
- Omzetten naar integer raster  
- Clippen van eindresultaat op het rekengebied  

**Output**
- Per polder een GeoTIFF met gecombineerd AHN en peil  
- Tijdelijke rasterbestanden voor tussenstapp


### 3.2 Achtergrondfunctie: Merge resulterende AHN met peil raster

Deze functie combineert het verwerkte AHN met het peilraster en voert een interpolatiestap uit om ontbrekende waarden te vullen.  
Het resultaat is een consistent en volledig rasterbestand per polder dat geschikt is voor verdere volumeberekeningen.


In [7]:
def export_ahn_peil_tifs_for_polders(
    rekengebieden_dict,
    peil_label,
    neighborhood,
    mosaic_method="MAXIMUM",
    cleanup=True
):
    
    """
    Exporteert per polder een definitieve GeoTIFF met
    gecombineerde AHN- en peilinformatie.

    Werkwijze
    ---------
    1. Lees het definitieve AHN-raster (ahn_final_<polder>).
    2. Lees het gerasteriseerde peilbestand (bijv. Peil_zo).
    3. Clip het AHN-raster op het rekengebied.
    4. Combineer AHN en peil via een mosaic-operatie.
    5. Interpoleer ontbrekende waarden met Focal Statistics.
    6. Converteer naar een integer raster.
    7. Clip het resultaat opnieuw op het rekengebied.
    8. Exporteer als GeoTIFF.

    Parameters
    ----------
    rekengebieden_dict : dict[str, str]
        Dictionary met per polder het pad naar het
        rekengebied.

    peil_label : str
        Naam van het peilraster, bijvoorbeeld:
        - 'Peil_zo'
        - 'Peil_wi'

    neighborhood : arcpy.sa.Nbr*
        Neighborhood-object dat wordt gebruikt
        voor de Focal Statistics interpolatie.

    mosaic_method : str, default='MAXIMUM'
        Methode voor het combineren van AHN en
        peilraster.

    cleanup : bool, default=True
        Verwijder tijdelijke TIFF-bestanden.

    Output
    ------
    Per polder wordt een GeoTIFF weggeschreven:

    <waterschap>_<polder>_<peil>.tif

    bijvoorbeeld:

    hhnk_beemster_Peil_zo.tif
    """

    def require_exists(path, label):
        if not arcpy.Exists(path):
            raise FileNotFoundError(f"--- {label} ontbreekt:\n{path} ---")

    for polder, rekengebied_fc in rekengebieden_dict.items():

        print(f"\n--- Export AHN + {peil_label} voor polder: {polder} ---")

        # --------------------------------------------------
        # Veilige namen
        # --------------------------------------------------
        safe_fs = maak_veilige_naam(
            polder,
            target="filesystem"
        )

        polder_gdb = arcpy.Describe(rekengebied_fc).path

        safe_gdb = maak_veilige_naam(
            polder,
            target="gdb_object",
            workspace=polder_gdb
        )

        # --------------------------------------------------
        # Paden
        # --------------------------------------------------
        polder_map = os.path.dirname(polder_gdb)
        waterschap = os.path.basename(os.path.dirname(polder_map))

        temp_gdb = os.path.join(
            polder_map, f"_temp_{safe_fs}.gdb"
        )

        # --------------------------------------------------
        # Workspace controle
        # --------------------------------------------------
        arcpy.ClearWorkspaceCache_management()

        if not arcpy.Exists(temp_gdb):
            raise FileNotFoundError(
                f"--- Temp GDB niet gevonden:\n{temp_gdb} ---"
            )

        arcpy.env.workspace = temp_gdb
        rasters = arcpy.ListRasters()

        peil_raster = os.path.join(temp_gdb, peil_label)
        
        if not arcpy.Exists(peil_raster):
            print(
                f"--- {peil_label} ontbreekt voor {polder} "
                "→ overslaan ---"
            )
            continue

        print(f"--- Temp GDB OK ({len(rasters)} raster(s)) ---")

        # --------------------------------------------------
        # Input datasets
        # --------------------------------------------------
        ahn_final = os.path.join(
            polder_gdb, f"ahn_final_{safe_gdb}"
        )
        peil_raster = os.path.join(
            temp_gdb, peil_label
        )

        require_exists(ahn_final, "AHN raster")
        require_exists(peil_raster, "Peil raster")

        # --------------------------------------------------
        # Output GeoTIFF
        # --------------------------------------------------
        out_name = f"{waterschap}_{safe_fs}_{peil_label}.tif"
        out_tif = os.path.join(polder_map, out_name)

        if arcpy.Exists(out_tif):
            print(f"--- Bestaat al → overslaan: {out_name} ---")
            continue

        # --------------------------------------------------
        # 1. AHN clippen op polder
        # --------------------------------------------------
        tmp_ahn_clip = os.path.join(
            polder_map, f"_tmp_ahn_{safe_fs}.tif"
        )

        ExtractByMask(
            ahn_final,
            rekengebied_fc
        ).save(tmp_ahn_clip)

        # --------------------------------------------------
        # 2. Mosaic AHN + peil
        # --------------------------------------------------
        tmp_mosaic = os.path.join(
            polder_map, f"_tmp_mosaic_{safe_fs}.tif"
        )

        if arcpy.Exists(tmp_mosaic):
            arcpy.management.Delete(tmp_mosaic)

        cellsize = arcpy.Describe(tmp_ahn_clip).meanCellWidth

        arcpy.management.MosaicToNewRaster(
            [tmp_ahn_clip, peil_raster],
            polder_map,
            os.path.basename(tmp_mosaic),
            pixel_type="32_BIT_SIGNED",
            cellsize=cellsize,
            number_of_bands=1,
            mosaic_method=mosaic_method
        )

        # --------------------------------------------------
        # 3. Interpolatie
        # --------------------------------------------------
        tmp_interp = os.path.join(
            polder_map, f"_tmp_interp_{safe_fs}.tif"
        )

        interp = FocalStatistics(
            tmp_mosaic,
            neighborhood,
            "MEAN",
            "DATA"
        )
        interp.save(tmp_interp)

        # --------------------------------------------------
        # 4. INTEGER + clip
        # --------------------------------------------------
        final_int = Int(tmp_interp)

        final_clipped = ExtractByMask(
            final_int,
            rekengebied_fc
        )

        arcpy.management.CopyRaster(
            in_raster=final_clipped,
            out_rasterdataset=out_tif,
            pixel_type="32_BIT_SIGNED"
        )

        print(f"--- Opgeslagen: {out_name} ---")

        # --------------------------------------------------
        # Cleanup tijdelijke bestanden
        # --------------------------------------------------
        if cleanup:
            for tmp in (tmp_ahn_clip, tmp_mosaic, tmp_interp):
                if arcpy.Exists(tmp):
                    try:
                        arcpy.management.Delete(tmp)
                    except Exception:
                        pass

    print("\n--- Export afgerond voor alle polders ---")


### 3.2 Invoer: merge resulterende AHN met peil raster

**Doel**  
Uitvoeren van de combinatie van het bewerkte AHN-raster met het peilraster per polder en exporteren naar GeoTIFF.

**Configuratie**
- **rekengebieden_dict**: dictionary met rekengebieden per polder  
- **peil_label**: naam van het peilraster (bijv. "Peil_zo" of "Peil_wi")  
- **neighborhood**: kernel voor interpolatie (bijv. 5x5 cellen)  
- **mosaic_method**: methode voor combineren rasters (bijv. "MAXIMUM")  
- **cleanup**: verwijderen van tijdelijke bestanden (True/False)  

**Output**
- Per polder een GeoTIFF met gecombineerd AHN en peil  

In [8]:
export_ahn_peil_tifs_for_polders(
    rekengebieden_dict=resultaten,
    peil_label="Peil_zo", 
    neighborhood=arcpy.sa.NbrRectangle(5, 5, "CELL"),# exact de rasternaam in _temp_<polder>.gdb
    mosaic_method="MAXIMUM",  # aanbevolen
    cleanup=True              # tijdelijke TIFFs opruimen
)


--- Export AHN + Peil_zo voor polder: Polder Holland en Sticht west ---
--- Temp GDB OK (1 raster(s)) ---
--- Opgeslagen: agv_polder_holland_en_sticht_west_Peil_zo.tif ---

--- Export AHN + Peil_zo voor polder: Westerpark ---
--- Temp GDB OK (1 raster(s)) ---
--- Opgeslagen: agv_westerpark_Peil_zo.tif ---

--- Export AHN + Peil_zo voor polder: Gansenhoef west ---
--- Temp GDB OK (1 raster(s)) ---
--- Opgeslagen: agv_gansenhoef_west_Peil_zo.tif ---

--- Export AHN + Peil_zo voor polder: Breukelen boezempeil ---
--- Temp GDB OK (1 raster(s)) ---
--- Opgeslagen: agv_breukelen_boezempeil_Peil_zo.tif ---

--- Export AHN + Peil_zo voor polder: De Lange Bretten ---
--- Temp GDB OK (1 raster(s)) ---
--- Opgeslagen: agv_de_lange_bretten_Peil_zo.tif ---

--- Export AHN + Peil_zo voor polder: Stadsboezem Amsterdam ---
--- Temp GDB OK (1 raster(s)) ---
--- Opgeslagen: agv_stadsboezem_amsterdam_Peil_zo.tif ---

--- Export AHN + Peil_zo voor polder: Noordpolder beoosten Muiden ---
--- Temp GDB OK (


--- Export AHN + Peil_zo voor polder: Oosterpark ---
--- Temp GDB OK (1 raster(s)) ---
--- Opgeslagen: agv_oosterpark_Peil_zo.tif ---

--- Export AHN + Peil_zo voor polder: Bijlmer ---
--- Temp GDB OK (1 raster(s)) ---
--- Opgeslagen: agv_bijlmer_Peil_zo.tif ---

--- Export AHN + Peil_zo voor polder: Gemeenschapspolder zuid-oost ---
--- Temp GDB OK (1 raster(s)) ---
--- Opgeslagen: agv_gemeenschapspolder_zuid_oost_Peil_zo.tif ---

--- Export AHN + Peil_zo voor polder: Buitendijksgebied Naarden en Muiderberg ---
--- Geen rasters in temp GDB voor Buitendijksgebied Naarden en Muiderberg → overslaan ---

--- Export AHN + Peil_zo voor polder: Buitendijks gebied Muiderberg ---
--- Geen rasters in temp GDB voor Buitendijks gebied Muiderberg → overslaan ---

--- Export AHN + Peil_zo voor polder: Bloemendalerpolder (noord) ---
--- Temp GDB OK (1 raster(s)) ---
--- Opgeslagen: agv_bloemendalerpolder_noord_Peil_zo.tif ---

--- Export AHN + Peil_zo voor polder: Noorder- of Rietpolder (De Krijgsma